In [2]:
# =====================================================
# EXPERIMENT 3
# PART 1: DATASET PREPARATION
# STANDARD 5-FOLD CROSS VALIDATION
# =====================================================

import pandas as pd
import numpy as np

print("="*70)
print("EXPERIMENT 3")
print("STANDARD 5-FOLD CROSS VALIDATION")
print("PART 1: DATASET PREPARATION")
print("="*70)


# =====================================================
# LOAD TON-IOT DATASET
# =====================================================

print("\nLoading TON-IoT Dataset")

dataset = pd.read_csv(
    "../data/train_test_network.csv"
)

print(
    "Dataset Shape:",
    dataset.shape
)


# =====================================================
# CLEAN DATASET
# =====================================================

print("\nCleaning Dataset")

dataset.replace(
    "-",
    "unknown",
    inplace=True
)

dataset.fillna(
    "unknown",
    inplace=True
)

print("Cleaning Completed")


# =====================================================
# DROP UNUSED FEATURES
# =====================================================

drop_columns = [

    "src_ip",
    "dst_ip",

    "src_port",
    "dst_port",

    "ssl_subject",
    "ssl_issuer",

    "http_user_agent",

    "label"

]

dataset.drop(

    columns=drop_columns,

    inplace=True,

    errors="ignore"

)


print("\nDataset Shape After Feature Removal")

print(
    dataset.shape
)


# =====================================================
# CLASS DISTRIBUTION
# =====================================================

print("\nClass Distribution")

print(
    dataset["type"].value_counts()
)


print("\n")

print("="*70)
print("PART 1 COMPLETED")
print("="*70)

EXPERIMENT 3
STANDARD 5-FOLD CROSS VALIDATION
PART 1: DATASET PREPARATION

Loading TON-IoT Dataset
Dataset Shape: (211043, 44)

Cleaning Dataset
Cleaning Completed

Dataset Shape After Feature Removal
(211043, 36)

Class Distribution
type
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
ransomware    20000
scanning      20000
xss           20000
mitm           1043
Name: count, dtype: int64


PART 1 COMPLETED


In [3]:
# =====================================================
# EXPERIMENT 3
# PART 2: DATA PREPROCESSING
# STANDARD 5-FOLD CROSS VALIDATION
# =====================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)

from sklearn.compose import ColumnTransformer

print("="*70)
print("EXPERIMENT 3")
print("PART 2: DATA PREPROCESSING")
print("="*70)


# =====================================================
# SEPARATE FEATURES AND LABEL
# =====================================================

print("\nSeparating Features and Labels")

X = dataset.drop(
    "type",
    axis=1
)

y = dataset["type"]

print("\nFeature Shape")

print(
    "X:",
    X.shape
)

print(
    "y:",
    y.shape
)


# =====================================================
# LABEL ENCODING
# =====================================================

print("\nEncoding Labels")

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(
    y
)

print("\nClasses")

print(
    label_encoder.classes_
)


# =====================================================
# IDENTIFY FEATURE TYPES
# =====================================================

print("\nIdentifying Feature Types")

categorical_columns = X.select_dtypes(
    include="object"
).columns.tolist()

numerical_columns = X.select_dtypes(
    exclude="object"
).columns.tolist()

print(
    "Categorical Features:",
    len(categorical_columns)
)

print(
    "Numerical Features:",
    len(numerical_columns)
)


# =====================================================
# FIX DATA TYPES
# =====================================================

print("\nFixing Data Types")


# categorical conversion

for col in categorical_columns:

    X[col] = X[col].astype(str)


# numerical conversion

for col in numerical_columns:

    X[col] = pd.to_numeric(

        X[col],

        errors="coerce"

    )


# replace invalid numerical values

X[numerical_columns] = (

    X[numerical_columns]

    .fillna(0)

)

print(
    "Datatype Correction Completed"
)


# =====================================================
# BUILD PREPROCESSOR
# =====================================================

print("\nBuilding Preprocessor")

preprocessor = ColumnTransformer(

    transformers=[

        (

            "cat",

            OneHotEncoder(

                handle_unknown="ignore",

                sparse_output=False

            ),

            categorical_columns

        ),

        (

            "num",

            StandardScaler(),

            numerical_columns

        )

    ]

)

print("\nPreprocessor Ready")

print("\n")

print("="*70)
print("PART 2 COMPLETED")
print("="*70)

EXPERIMENT 3
PART 2: DATA PREPROCESSING

Separating Features and Labels

Feature Shape
X: (211043, 35)
y: (211043,)

Encoding Labels

Classes
['backdoor' 'ddos' 'dos' 'injection' 'mitm' 'normal' 'password'
 'ransomware' 'scanning' 'xss']

Identifying Feature Types
Categorical Features: 21
Numerical Features: 14

Fixing Data Types
Datatype Correction Completed

Building Preprocessor

Preprocessor Ready


PART 2 COMPLETED


C:\Users\Kruthik Raj M\AppData\Local\Temp\ipykernel_23272\2801513258.py:75: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


In [6]:
# =====================================================
# EXPERIMENT 3
# PART 3: RANDOM FOREST
# 5-FOLD CROSS VALIDATION
# CLASSIFICATION REPORT
# =====================================================

import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report

)


print("="*70)
print("EXPERIMENT 3")
print("PART 3: RANDOM FOREST")
print("5-FOLD CROSS VALIDATION")
print("="*70)



# =====================================================
# STRATIFIED 5-FOLD
# =====================================================

skf = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)



# =====================================================
# STORE METRICS
# =====================================================

accuracy_scores = []

precision_scores = []

recall_scores = []

weighted_f1_scores = []

macro_f1_scores = []


# Store all predictions

all_y_true = []

all_y_pred = []



# =====================================================
# CROSS VALIDATION
# =====================================================

for fold, (train_index, valid_index) in enumerate(

    skf.split(X, y_encoded),

    start=1

):

    print("\n" + "="*60)

    print(f"Fold {fold}")

    print("="*60)



    X_train = X.iloc[train_index].copy()

    X_valid = X.iloc[valid_index].copy()


    y_train = y_encoded[train_index]

    y_valid = y_encoded[valid_index]



    # Preprocessing inside fold

    X_train_processed = preprocessor.fit_transform(

        X_train

    )


    X_valid_processed = preprocessor.transform(

        X_valid

    )



    # Random Forest

    rf = RandomForestClassifier(

        n_estimators=200,

        random_state=42,

        n_jobs=-1

    )



    rf.fit(

        X_train_processed,

        y_train

    )



    predictions = rf.predict(

        X_valid_processed

    )



    # Save predictions

    all_y_true.extend(y_valid)

    all_y_pred.extend(predictions)



    # Metrics

    accuracy = accuracy_score(

        y_valid,

        predictions

    )


    precision = precision_score(

        y_valid,

        predictions,

        average="weighted",

        zero_division=0

    )


    recall = recall_score(

        y_valid,

        predictions,

        average="weighted",

        zero_division=0

    )


    weighted_f1 = f1_score(

        y_valid,

        predictions,

        average="weighted",

        zero_division=0

    )


    macro_f1 = f1_score(

        y_valid,

        predictions,

        average="macro",

        zero_division=0

    )


    accuracy_scores.append(accuracy)

    precision_scores.append(precision)

    recall_scores.append(recall)

    weighted_f1_scores.append(weighted_f1)

    macro_f1_scores.append(macro_f1)



    print(f"Accuracy     : {accuracy:.4f}")

    print(f"Precision    : {precision:.4f}")

    print(f"Recall       : {recall:.4f}")

    print(f"Weighted F1  : {weighted_f1:.4f}")

    print(f"Macro F1     : {macro_f1:.4f}")





# =====================================================
# FINAL CV RESULTS
# =====================================================

print("\n")

print("="*70)

print("RANDOM FOREST")

print("5-FOLD CROSS VALIDATION RESULTS")

print("="*70)


print(
    f"Mean Accuracy : {np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}"
)

print(
    f"Mean Precision : {np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}"
)

print(
    f"Mean Recall : {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}"
)

print(
    f"Mean Weighted F1 : {np.mean(weighted_f1_scores):.4f} ± {np.std(weighted_f1_scores):.4f}"
)

print(
    f"Mean Macro F1 : {np.mean(macro_f1_scores):.4f} ± {np.std(macro_f1_scores):.4f}"
)



# =====================================================
# CLASSIFICATION REPORT
# =====================================================

print("\n")

print("="*70)

print("RANDOM FOREST")

print("5-FOLD CROSS VALIDATION CLASSIFICATION REPORT")

print("="*70)



print(

    classification_report(

        all_y_true,

        all_y_pred,

        target_names=label_encoder.classes_,

        digits=4,

        zero_division=0

    )

)

EXPERIMENT 3
PART 3: RANDOM FOREST
5-FOLD CROSS VALIDATION

Fold 1
Accuracy     : 0.9783
Precision    : 0.9787
Recall       : 0.9783
Weighted F1  : 0.9783
Macro F1     : 0.9577

Fold 2
Accuracy     : 0.9793
Precision    : 0.9797
Recall       : 0.9793
Weighted F1  : 0.9794
Macro F1     : 0.9551

Fold 3
Accuracy     : 0.9775
Precision    : 0.9780
Recall       : 0.9775
Weighted F1  : 0.9776
Macro F1     : 0.9553

Fold 4
Accuracy     : 0.9772
Precision    : 0.9778
Recall       : 0.9772
Weighted F1  : 0.9773
Macro F1     : 0.9568

Fold 5
Accuracy     : 0.9778
Precision    : 0.9785
Recall       : 0.9778
Weighted F1  : 0.9780
Macro F1     : 0.9563


RANDOM FOREST
5-FOLD CROSS VALIDATION RESULTS
Mean Accuracy : 0.9780 ± 0.0007
Mean Precision : 0.9785 ± 0.0006
Mean Recall : 0.9780 ± 0.0007
Mean Weighted F1 : 0.9781 ± 0.0007
Mean Macro F1 : 0.9562 ± 0.0010


RANDOM FOREST
5-FOLD CROSS VALIDATION CLASSIFICATION REPORT
              precision    recall  f1-score   support

    backdoor     0.9996 

In [7]:
# =====================================================
# EXPERIMENT 3
# PART 4: XGBOOST
# 5-FOLD CROSS VALIDATION
# CLASSIFICATION REPORT
# =====================================================

import numpy as np

from sklearn.model_selection import StratifiedKFold

from xgboost import XGBClassifier

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report

)


print("="*70)
print("EXPERIMENT 3")
print("PART 4: XGBOOST")
print("5-FOLD CROSS VALIDATION")
print("="*70)



# =====================================================
# STRATIFIED 5-FOLD
# =====================================================

skf = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)



# =====================================================
# STORE METRICS
# =====================================================

accuracy_scores = []

precision_scores = []

recall_scores = []

weighted_f1_scores = []

macro_f1_scores = []



# Store predictions from all folds

all_y_true = []

all_y_pred = []



# =====================================================
# CROSS VALIDATION
# =====================================================

for fold, (train_index, valid_index) in enumerate(

    skf.split(X, y_encoded),

    start=1

):

    print("\n" + "="*60)

    print(f"Fold {fold}")

    print("="*60)



    # -------------------------------------------------
    # SPLIT DATA
    # -------------------------------------------------

    X_train = X.iloc[train_index].copy()

    X_valid = X.iloc[valid_index].copy()


    y_train = y_encoded[train_index]

    y_valid = y_encoded[valid_index]



    # -------------------------------------------------
    # FIT PREPROCESSOR ONLY ON TRAINING FOLD
    # -------------------------------------------------

    X_train_processed = preprocessor.fit_transform(

        X_train

    )


    X_valid_processed = preprocessor.transform(

        X_valid

    )



    # -------------------------------------------------
    # BUILD XGBOOST MODEL
    # SAME HYPERPARAMETERS
    # -------------------------------------------------

    xgb = XGBClassifier(

        n_estimators=200,

        learning_rate=0.1,

        random_state=42,

        n_jobs=-1,

        objective="multi:softprob",

        eval_metric="mlogloss"

    )



    # -------------------------------------------------
    # TRAIN MODEL
    # -------------------------------------------------

    xgb.fit(

        X_train_processed,

        y_train

    )



    # -------------------------------------------------
    # PREDICTION
    # -------------------------------------------------

    predictions = xgb.predict(

        X_valid_processed

    )



    # -------------------------------------------------
    # STORE ALL PREDICTIONS
    # -------------------------------------------------

    all_y_true.extend(

        y_valid

    )


    all_y_pred.extend(

        predictions

    )



    # -------------------------------------------------
    # METRICS
    # -------------------------------------------------

    accuracy = accuracy_score(

        y_valid,

        predictions

    )


    precision = precision_score(

        y_valid,

        predictions,

        average="weighted",

        zero_division=0

    )


    recall = recall_score(

        y_valid,

        predictions,

        average="weighted",

        zero_division=0

    )


    weighted_f1 = f1_score(

        y_valid,

        predictions,

        average="weighted",

        zero_division=0

    )


    macro_f1 = f1_score(

        y_valid,

        predictions,

        average="macro",

        zero_division=0

    )



    # -------------------------------------------------
    # SAVE RESULTS
    # -------------------------------------------------

    accuracy_scores.append(

        accuracy

    )


    precision_scores.append(

        precision

    )


    recall_scores.append(

        recall

    )


    weighted_f1_scores.append(

        weighted_f1

    )


    macro_f1_scores.append(

        macro_f1

    )



    # -------------------------------------------------
    # PRINT FOLD RESULTS
    # -------------------------------------------------

    print(f"Accuracy     : {accuracy:.4f}")

    print(f"Precision    : {precision:.4f}")

    print(f"Recall       : {recall:.4f}")

    print(f"Weighted F1  : {weighted_f1:.4f}")

    print(f"Macro F1     : {macro_f1:.4f}")




# =====================================================
# FINAL CROSS VALIDATION RESULTS
# =====================================================

print("\n")

print("="*70)

print("XGBOOST")

print("5-FOLD CROSS VALIDATION RESULTS")

print("="*70)



print(

    f"Mean Accuracy     : {np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}"

)


print(

    f"Mean Precision    : {np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}"

)


print(

    f"Mean Recall       : {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}"

)


print(

    f"Mean Weighted F1  : {np.mean(weighted_f1_scores):.4f} ± {np.std(weighted_f1_scores):.4f}"

)


print(

    f"Mean Macro F1     : {np.mean(macro_f1_scores):.4f} ± {np.std(macro_f1_scores):.4f}"

)




# =====================================================
# CLASSIFICATION REPORT
# =====================================================

print("\n")

print("="*70)

print("XGBOOST")

print("5-FOLD CROSS VALIDATION CLASSIFICATION REPORT")

print("="*70)



print(

    classification_report(

        all_y_true,

        all_y_pred,

        target_names=label_encoder.classes_,

        digits=4,

        zero_division=0

    )

)



print("\n")

print("="*70)

print("PART 4 COMPLETED")

print("="*70)

EXPERIMENT 3
PART 4: XGBOOST
5-FOLD CROSS VALIDATION

Fold 1
Accuracy     : 0.9787
Precision    : 0.9795
Recall       : 0.9787
Weighted F1  : 0.9789
Macro F1     : 0.9565

Fold 2
Accuracy     : 0.9800
Precision    : 0.9807
Recall       : 0.9800
Weighted F1  : 0.9802
Macro F1     : 0.9546

Fold 3
Accuracy     : 0.9784
Precision    : 0.9792
Recall       : 0.9784
Weighted F1  : 0.9786
Macro F1     : 0.9562

Fold 4
Accuracy     : 0.9779
Precision    : 0.9788
Recall       : 0.9779
Weighted F1  : 0.9781
Macro F1     : 0.9529

Fold 5
Accuracy     : 0.9785
Precision    : 0.9793
Recall       : 0.9785
Weighted F1  : 0.9787
Macro F1     : 0.9551


XGBOOST
5-FOLD CROSS VALIDATION RESULTS
Mean Accuracy     : 0.9787 ± 0.0007
Mean Precision    : 0.9795 ± 0.0007
Mean Recall       : 0.9787 ± 0.0007
Mean Weighted F1  : 0.9789 ± 0.0007
Mean Macro F1     : 0.9551 ± 0.0013


XGBOOST
5-FOLD CROSS VALIDATION CLASSIFICATION REPORT
              precision    recall  f1-score   support

    backdoor     0.9998 